# 14 — Orquestrador `executar_pipeline`

Desenvolve o `principal` que liga a esteira. **F10, NF5.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np
import pandas as pd
from app import dal, nucleo
from app.mercado import RendaFixa, RendaVariavel
from app.agente import Investidor

## Desenvolvimento

A função abaixo foi escrita aqui e, após os testes, movida para `app/principal.py`.

In [2]:
# Fator de desconto ANUAL padrao.
# Fica declarado ao ano porque "beta = 0.96" sozinho nao diz a que periodo se
# refere. Lido como valor por pregao, vira 0.96 elevado a 252, quase zero, e o
# investidor consumiria quase tudo no primeiro ano.
BETA_ANUAL_PADRAO = 0.96

In [3]:
def _n_scenarios_padrao(periodos_por_ano: int) -> int:
    """Cenarios de Monte Carlo adequados a frequencia dos dados.

    A base diaria precisa de muito mais, porque o excesso de retorno de um
    pregao e pequeno perto do desvio-padrao dele e com poucos cenarios o
    alpha* oscila de uma rodada pra outra.
    """
    return 200_000 if periodos_por_ano <= 12 else 4_000_000

In [ ]:
def _consumo_por_ano(c_medio: np.ndarray, periodos_por_ano: int, T: int) -> np.ndarray:
    """Soma o consumo medio dentro de cada ano do horizonte."""
    n_anos = max(1, int(np.ceil(T / periodos_por_ano))) 
    fluxo = np.asarray(c_medio, dtype=float)[:T] 
    return np.array([fluxo[i * periodos_por_ano:(i + 1) * periodos_por_ano].sum()
                     for i in range(n_anos)])

In [ ]:
def executar_pipeline(config: dict) -> dict:
    """Roda a esteira inteira e devolve os resultados. (NF5)

    O que pode vir no config:

    periodos_por_ano: 12 se a base for mensal, 252 se for diaria. E
        obrigatorio: sem ele nao da pra converter o cdi_anual nem o
        beta_anual, que sao declarados ao ano.
    retornos ou db_path: ou o DataFrame ja pronto, com a coluna data mais uma
        coluna por ativo, ou o caminho do banco pra ler do SQLite (nesse caso
        da pra passar tambem tabela, que por padrao e 'retornos').
    ativos: quais colunas sao de risco. Por padrao, todas menos a data e a
        coluna do rf.
    rf_col: nome da coluna da taxa livre de risco nos dados, por padrao 'cdi'.
    cdi_anual: o CDI ao ano, convertido pro periodo aqui dentro. Se nao vier,
        usa a media da coluna do rf, que o dal ja grava por periodo.
    gamma (5.0), w0 (1.0) e horizonte, que e o T em periodos e tambem e
        obrigatorio.
    beta_anual (0.96, convertido pro periodo) ou beta, pra quem ja tiver o
        valor por periodo. Passar os dois da erro.
    n_scenarios (200 mil no mensal, 4 milhoes no diario), n_paths (5000, mas a
        linha de comando pede 3000) e seed (42).

    O que volta: um dicionario com alpha_star (a carteira otima), theta e
    consumo_inicial, phi_hat, A_t (Etapa 3), a calibracao (mu_hat, sigma_hat e
    rf) e o resumo da simulacao, com E_W_T, os percentis de W_T (W_T_p5 e
    W_T_p95) e as trajetorias
    trajetoria_W_media, _mediana, _p5, _p95 e trajetoria_c_media, mais o
    consumo_por_ano ja somado dentro de cada ano e dividido por w0, que e a
    fracao da riqueza inicial consumida no ano.
    """
    cfg = dict(config)
    coluna_data = cfg.get("coluna_data", "data")
    rf_col = cfg.get("rf_col", "cdi")

    # 0. a frequencia, que da a unidade de tempo de todo o resto
    if "periodos_por_ano" not in cfg:
        raise ValueError(
            "falta 'periodos_por_ano' no config (12 se mensal, 252 se diario). "
            "Sem ele nao da pra converter 'cdi_anual' e 'beta_anual', que sao "
            "declarados ao ano."
        )
    ppa = int(cfg["periodos_por_ano"])
    if ppa <= 0:
        raise ValueError(f"'periodos_por_ano' deve ser positivo; veio {ppa!r}.")

    # 1. dal: pegar os retornos
    if cfg.get("retornos") is not None:
        retornos = cfg["retornos"]
    elif "db_path" in cfg:
        retornos = dal.ler_sqlite(cfg["db_path"], cfg.get("tabela", "retornos"))
    else:
        raise ValueError("config precisa de 'retornos' (DataFrame) ou 'db_path'.")

    colunas = [c for c in retornos.columns if c != coluna_data]
    ativos = cfg.get("ativos") or [c for c in colunas if c != rf_col]
    if not ativos:
        raise ValueError("nenhum ativo de risco identificado em 'retornos'.")

    # 2. mercado: a calibracao (Etapa 0)
    if "cdi_anual" in cfg:
        rf = RendaFixa(cfg["cdi_anual"], ppa).retorno_livre_risco()
    elif rf_col in retornos.columns:
        rf = float(retornos[rf_col].mean())
    else:
        rf = float(cfg.get("rf", 0.0))
    mercado = RendaVariavel(retornos[[coluna_data] + ativos], coluna_data=coluna_data)

    # 3. agente: a politica otima (Etapas 1 a 4)
    if "horizonte" not in cfg:
        raise ValueError(
            "falta 'horizonte' no config (o T, contado em periodos). Nao tem "
            "padrao porque 60 periodos e 5 anos no mensal e uns 3 meses no diario."
        )
    if "beta" in cfg and "beta_anual" in cfg:
        raise ValueError("use 'beta' (por periodo) OU 'beta_anual', nao os dois.")
    beta = (float(cfg["beta"]) if "beta" in cfg
            else float(cfg.get("beta_anual", BETA_ANUAL_PADRAO)) ** (1.0 / ppa))
    inv = Investidor(cfg.get("gamma", 5.0), beta,
                     cfg.get("w0", 1.0), cfg["horizonte"])
    seed = cfg.get("seed", 42)
    n_scenarios = cfg.get("n_scenarios") or _n_scenarios_padrao(ppa)
    alpha = inv.carteira_otima(mercado, rf, n_scenarios=n_scenarios, seed=seed)
    theta = inv.fracoes_consumo()

    # 4. simulacao pra frente (Etapas 5 e 6)
    T, N = inv.horizonte, len(ativos)
    n_paths = cfg.get("n_paths", 5_000)
    r_paths = mercado.amostrar(n_paths * T, seed=seed + 1)
    R_paths = np.maximum(1.0 + r_paths.reshape(n_paths, T, N), 0.0)
    sim = nucleo.propagar_riqueza(inv.w0, theta, alpha, R_paths, 1.0 + rf)

    # 5. o resultado
    W_T = sim["W"][:, -1]
    A_t = inv.coeficientes_A
    B_t = (nucleo.recorrencia_B(A_t, inv.phi_hat, beta)
           if np.isclose(inv.gamma, 1.0) else None)
    V = nucleo.funcao_valor(A_t, sim["W"], inv.gamma, B=B_t)
    W_p5, W_p50, W_p95 = np.percentile(sim["W"], [5, 50, 95], axis=0)
    return {
        "ativos": ativos,
        "periodos_por_ano": ppa,
        "rf": rf,
        "beta": beta,
        "mu_hat": mercado.media(),
        "sigma_hat": mercado.covariancia(),
        "alpha_star": alpha,
        "phi_hat": inv.phi_hat,
        "theta": theta,
        "consumo_inicial": float(theta[0] * inv.w0),
        "horizonte": T,
        "E_W_T": float(W_T.mean()),
        "W_T_p5": float(np.percentile(W_T, 5)),
        "W_T_p95": float(np.percentile(W_T, 95)),
        "A_t": A_t,
        "trajetoria_W_media": sim["W"].mean(axis=0),
        "trajetoria_W_mediana": W_p50,
        "trajetoria_W_p5": W_p5,
        "trajetoria_W_p95": W_p95,
        "trajetoria_c_media": sim["c"].mean(axis=0),
        "trajetoria_V_media": V.mean(axis=0),
        "trajetoria_u_media": inv.utilidade(sim["c"]).mean(axis=0),
        "consumo_por_ano": _consumo_por_ano(sim["c"].mean(axis=0) / inv.w0, ppa, T),
    }

**Teste**: roda a esteira e devolve resultado coerente.

In [6]:
import pandas as pd

In [7]:
rng = np.random.default_rng(7)
ruido = rng.normal(0,0.06,200)
ruido -= ruido.mean()
ret = pd.DataFrame({'data': pd.date_range('2000-01',periods=200,freq='MS').strftime('%Y-%m'),'ibov':0.015+ruido,'cdi':np.full(200,0.008)})
res = executar_pipeline({'retornos':ret,'ativos':['ibov'],'periodos_por_ano':12,
                        'cdi_anual':0.10,'gamma':5.0,'beta':0.96,'w0':1.0,'horizonte':12,'n_scenarios':40_000,'n_paths':2_000,'seed':1})

print('chaves:', list(res.keys())); print('alpha*:', res['alpha_star'], '| E[W_T]:', res['E_W_T'])

chaves: ['ativos', 'periodos_por_ano', 'rf', 'beta', 'mu_hat', 'sigma_hat', 'alpha_star', 'phi_hat', 'theta', 'consumo_inicial', 'horizonte', 'E_W_T', 'W_T_p5', 'W_T_p95', 'A_t', 'trajetoria_W_media', 'trajetoria_W_mediana', 'trajetoria_W_p5', 'trajetoria_W_p95', 'trajetoria_c_media', 'trajetoria_V_media', 'trajetoria_u_media', 'consumo_por_ano']
alpha*: [0.48558069] | E[W_T]: 0.08019876896545776


In [8]:
assert res['alpha_star'].shape==(1,) and np.isclose(res['theta'][-1],1.0) and res['E_W_T']>0

**Teste**: as médias da Etapa 7. Em t = 0 a média de V_t(W_t) tem que dar o próprio V_0(W_0), porque todo caminho começa em W_0, e em t = T tem que dar a média de u(c_T), porque aí a riqueza é toda consumida. Com gamma = 1 vale o mesmo, somando o B_t.

In [9]:
res_log = executar_pipeline({'retornos':ret,'ativos':['ibov'],'periodos_por_ano':12,
                             'cdi_anual':0.10,'gamma':1.0,'beta':0.96,'w0':1.0,'horizonte':12,
                             'n_scenarios':40_000,'n_paths':2_000,'seed':1})
B_log = nucleo.recorrencia_B(res_log['A_t'], res_log['phi_hat'], res_log['beta'])
V0_log = nucleo.funcao_valor(res_log['A_t'][:1], 1.0, 1.0, B=B_log[:1])[0]

print('gamma=5: E[V_0] =', res['trajetoria_V_media'][0], '| E[V_T] =', res['trajetoria_V_media'][-1])
print('gamma=1: E[V_0] =', res_log['trajetoria_V_media'][0], '| V_0 com B_0 =', V0_log)

gamma=5: E[V_0] = -58326.30139892972 | E[V_T] = -6510.016487725745
gamma=1: E[V_0] = -25.40539276653903 | V_0 com B_0 = -25.40539276653922


In [10]:
for r_, V0 in [(res, nucleo.funcao_valor(res['A_t'][:1], 1.0, 5.0)[0]), (res_log, V0_log)]:
    assert r_['trajetoria_V_media'].shape == (r_['horizonte'] + 1,) == r_['trajetoria_u_media'].shape
    assert np.isclose(r_['trajetoria_V_media'][0], V0)
    assert np.isclose(r_['trajetoria_V_media'][-1], r_['trajetoria_u_media'][-1])

**Teste**: a unidade de tempo não pode ser adivinhada: `periodos_por_ano` é obrigatório e é ele que converte `cdi_anual` e `beta_anual`.

In [11]:
# 1) sem 'periodos_por_ano' a esteira para, sem herdar 12 em silencio
base = {'retornos': ret, 'ativos': ['ibov'], 'horizonte': 12, 'n_scenarios': 20_000,
        'n_paths': 500, 'seed': 1}
try:
    executar_pipeline(base); raise SystemExit('deveria ter falhado')
except ValueError as e:
    assert 'periodos_por_ano' in str(e)

In [12]:
# 2) o mesmo cdi_anual gera R_f diferente em cada frequencia (e nao o mensal nas duas)
mensal = executar_pipeline({**base, 'periodos_por_ano': 12,  'cdi_anual': 0.1312})
diario = executar_pipeline({**base, 'periodos_por_ano': 252, 'cdi_anual': 0.1312})
assert np.isclose(mensal['rf'], 1.1312 ** (1 / 12) - 1)
assert np.isclose(diario['rf'], 1.1312 ** (1 / 252) - 1)

In [13]:
# 3) beta_anual vira beta do periodo; 'beta' cru continua aceito para quem ja converteu
assert np.isclose(diario['beta'], 0.96 ** (1 / 252))
assert np.isclose(executar_pipeline({**base, 'periodos_por_ano': 252,
                                     'beta': 0.5})['beta'], 0.5)

In [14]:
try:
    executar_pipeline({**base, 'periodos_por_ano': 12, 'beta': 0.9, 'beta_anual': 0.9})
    raise SystemExit('deveria ter falhado')
except ValueError as e:
    assert 'beta_anual' in str(e)

print('R_f mensal:', mensal['rf'], '| R_f diario:', diario['rf'])

R_f mensal: 0.010326202364327575 | R_f diario: 0.0004893221241111245


**Teste**: o consumo_por_ano soma por ano sem o c_T, que é a liquidação terminal, e deixa o último balde parcial quando T não fecha um número inteiro de anos.

In [15]:
base_cpa = {'retornos': ret, 'ativos': ['ibov'], 'periodos_por_ano': 12,
            'n_scenarios': 20_000, 'n_paths': 300, 'seed': 1}

cheio = executar_pipeline({**base_cpa, 'horizonte': 24})
quebr = executar_pipeline({**base_cpa, 'horizonte': 30})

c = quebr['trajetoria_c_media']
manual = [c[0:12].sum(), c[12:24].sum(), c[24:30].sum()]

print('T=24 ->', np.round(cheio['consumo_por_ano'], 4))
print('T=30 ->', np.round(quebr['consumo_por_ano'], 4), '(o 3o ano e parcial)')

T=24 -> [0.5407 0.5633]
T=30 -> [0.4464 0.4651 0.2392] (o 3o ano e parcial)


In [16]:
assert cheio['consumo_por_ano'].shape == (2,)
assert quebr['consumo_por_ano'].shape == (3,)
assert np.allclose(quebr['consumo_por_ano'], manual)
# o c_T fica de fora: a soma dos anos e menor que a soma da trajetoria inteira
assert quebr['consumo_por_ano'].sum() < c.sum()
assert np.isclose(quebr['consumo_por_ano'].sum(), c[:30].sum())

**Teste**: o consumo_por_ano é fração de W_0, então tem que dar o mesmo com W_0 = 1 e com W_0 = 2. O consumo em si dobra, a fração não muda.

In [17]:
w0_1 = executar_pipeline({**base_cpa, 'horizonte': 24, 'w0': 1.0})
w0_2 = executar_pipeline({**base_cpa, 'horizonte': 24, 'w0': 2.0})

print('W_0=1 ->', np.round(w0_1['consumo_por_ano'], 4), '| W_0=2 ->', np.round(w0_2['consumo_por_ano'], 4))

W_0=1 -> [0.5407 0.5633] | W_0=2 -> [0.5407 0.5633]


In [18]:
assert np.allclose(w0_1['consumo_por_ano'], w0_2['consumo_por_ano'])
assert np.allclose(w0_2['trajetoria_c_media'], 2 * w0_1['trajetoria_c_media'])